In [81]:
!python --version

Python 3.12.5


In [4]:
import random

## CONSTANTS

In [33]:
ONES = 0
TWOS = 1
THREES = 2
FOURS = 3
FIVES = 4
SIXES = 5
AAA = 6
AAAA = 7
AAABB = 8
ABCDE = 9
YAZY = 10

# CLASSES

## 1.  DICE CLASS

In [117]:
class Dice:
    def __init__(self):
        self.values = [0 for _ in range(5)]
        self.locked = [False for _ in range(5)]

    def roll(self):

        for i in range(5):
            if not self.locked[i]:
                self.values[i] = random.randint(1,6)

        return self.values

    def set_locked(self, prompt):

        if prompt == [-1]:
            for i in range(5):
                self.locked[i] = True
        
        else:
            for i in range(5):
                
                if i in prompt:
                    self.locked[i] = True
                else:
                    self.locked[i] = False

    def sum(self):
        return sum(self.values)

    def frequency(self):

        map = [0 for i in range(6)]

        for i in range(5):
            map[self.values[i] - 1] += 1

        frequency = {idx : map[idx - 1] for idx in range(1,7)}
        return frequency

    def sorted(self):
        return sorted(self.values)

    def reset(self):
        self.values = [0 for i in range(5)]
        self.locked = [False for i in range(5)]

## 2. BOARD CLASS

In [40]:
class Board:
    def __init__(self):
        self.scores = [0 for i in range(11)]
        self.taken = [False for i in range(11)]

    def pick_category(self, idx, score):
        self.scores[idx] = score
        self.taken[idx] = True

## 3. RULES CLASS

In [128]:
class Rules:

    def validateChoicePrompt(self, prompt):
        if prompt not in [1,2]:
            return False
        return True

    def validateSetLocked(self, prompt):
        if (len(prompt) >= 5):
            return False            

        for num in prompt:
            if num not in [1,2,3,4,5]:
                return False

        if len(set(prompt)) != len(prompt):
            return False

        return True

    def validatePickCategory(self, prompt):
        if prompt not in range(1,12):
            return False
        return True

    def calculateCategoryScore(self, idx, dice, board):

        if board.taken[idx]:
            return -1
        
        if idx in range(6):
            return dice.frequency()[idx + 1] * (idx + 1)
        if idx == AAA:
            if max(dice.frequency().values()) >= 3:
                return dice.sum()
            return 0
        if idx == AAAA:
            if max(dice.frequency().values()) >= 4:
                return dice.sum()
            return 0
        if idx == ABCDE:
            if dice.sorted() == [1,2,3,4,5] or dice.sorted() == [2,3,4,5,6]:
                return 40
            else:
                return 0
        if idx == AAABB:
            if 2 in dice.frequency().values() and 3 in dice.frequency().values():
                return 25
            else:
                return 0
        if idx == YAZY:
            if max(dice.frequency().values()) == 5:
                return 50
            else:
                return 0

    def calculateAvailableScores(self, dice, board):
        return [self.calculateCategoryScore(i, dice, board) for i in range(11)]
        

## 4. GAME CLASS

In [144]:
class Game:

    def __init__(self):

        self.state = GameState()

        self.rules = Rules()
        self.printer = Printer()

    def play_turn(self):

        self.state.numRolls = 1
        while self.state.numRolls <= 3:

            self.state.dice.roll()
            self.printer.print_roll(self.state.dice, self.state.numRolls)

            availableScores = self.rules.calculateAvailableScores(self.state.dice, self.state.board)
            self.printer.print_available_scores(self.state.board, availableScores)

            if self.state.numRolls != 3:
                choice = self.printer.input_choice_prompt(self.rules)

                if choice == 1:
                    prompt = self.printer.input_set_locked(self.rules)

                    prompt = [prompt[i] -1 for i in range(len(prompt))]
                    self.state.dice.set_locked(prompt)

                        
                else:
                    prompt = self.printer.input_pick_category(self.rules)
                    prompt -= 1
                    
                    self.state.board.pick_category(prompt, availableScores[prompt])
                    self.state.total_score += availableScores[prompt]
                    break

            else: 
                    prompt = self.printer.input_pick_category(self.rules)
                    prompt -= 1
                    
                    self.state.board.pick_category(prompt, availableScores[prompt])
                    self.state.total_score += availableScores[prompt]
                    break
                
            self.state.numRolls += 1
                
        self.state.numTurns += 1
    
    def run(self):

        self.state.dice.reset()
        
        while self.state.numTurns <= 11:
            self.printer.print_turn_info(self.state.numTurns, self.state.total_score)
            self.play_turn()

            self.state.dice.reset()

        self.state.is_game_over = True
        self.printer.print_game_over(self.state.board, self.state.total_score)

## 5. PRINTER CLASS

In [143]:
class Printer:
    
    def print_turn_info(self, numTurns, score):
        print("=" * 40)

        temp = f"TURN {numTurns :2}"
        print(f"{temp :^40}")
        print("=" * 40)

        print()
        temp = f"CURRENT SCORE: {score}"
        print(f"{temp :^40}")
        
    def print_roll(self, dice, numRoll):
        temp = f" ROLL {numRoll}/3 "
        print (f"{temp :-^40}")
        
        print(" "*12 + "DICE: ", end = "  ")
        for i in range(5):
            print(dice.values[i], end = " ")

        print()

    def print_available_scores(self, board, availableScores):

        temp = f" AVAILABLE SCORES "
        print (f"{temp:-^40}")
        
        categories = [
            "Ones   (1)",
            "Twos   (2)",
            "Threes (3)",
            "Fours  (4)",
            "Fives  (5)",
            "Sixes  (6)",
            "AAA",
            "AAAA",
            "AAABB",
            "ABCDE",
            "AAAAA"
        ]
    
        for i in range(11):
            if not board.taken[i]:
                temp = f"{i+1 :2}. {categories[i]:10} : {availableScores[i] : ^10}"
                print(f"{temp : ^40}")
            else:
                temp = f"{i+1 :2}. {categories[i]:10} : TAKEN ({board.scores[i]})"
                print(f"{temp : ^40}")

    def input_choice_prompt(self,rules):

        print("    Possible Actions:")
        
        print()
        
        print("    \t1. Roll Again")
        print("    \t2. Pick a Category")
        
        print()
        
        prompt = int(input("    Enter your choice here: "))

        while not rules.validateChoicePrompt(prompt):
            prompt = int(input("    Invalid Prompt!! Try Again: "))

        return prompt

    def input_set_locked(self, rules):

        print("    Choose which dice to lock")
        print("    \t1. The values should be space separated")
        print("    \t2. For rerolling all dice, enter 0")
        
        print()

        prompt = list(map(int,input("    Enter your choice here: ").split()))

        while not rules.validateSetLocked(prompt):
            prompt = list(map(int,input("    Invalid Choice!! Try Again: ").split()))

        return prompt

    def input_pick_category(self, rules):

        print("    Choose a category")
        print()

        prompt = int(input("    Enter your choice here: "))
        while not rules.validatePickCategory(prompt):
            prompt = int(input("    Enter your choice here: "))     
        return prompt

    def print_game_over(self, board, total_score):

        print("\n")
        print("=" * 40)
        print(f" {"Y A Z Y   -   G A M E   O V E R"} :^40 ")
        print("=" * 40)
        print()
    
        print(f"{'FINAL SCORE :':<30}{total_score:>10}")
        print()
    
        print("=" * 40)
        print(f"{'CATEGORY':<30}{'SCORE':>10}")
        print("-" * 40)
    
        categories = [
            "Ones",
            "Twos",
            "Threes",
            "Fours",
            "Fives",
            "Sixes",
            "AAA",
            "AAAA",
            "AAABB",
            "ABCDE",
            "YAZY"
        ]
    
        for i in range(11):
            print(f"{categories[i]:<30}{board.scores[i]:>10}")
    
        print("-" * 40)
        print(f"{'TOTAL':<30}{total_score:>10}")
        print("=" * 40)

## 6. GAMESTATE CLASS

In [142]:
class GameState:

    def __init__(self):
        self.dice = Dice()
        self.board = Board()

        self.numTurns = 1
        self.numRolls = 1
        self.total_score = 0

        self.is_game_over = False